# 03 — Ray Casting Algorithm

Here is the core idea, expressed as simply as possible:

```text
From the point, shoot a ray in any direction.
Count how many times the ray crosses the polygon boundary.

Odd count  →  inside
Even count →  outside
```

That's it. Everything else is implementation detail.

## Setup

In [ ]:
import json
from pathlib import Path
from ipyleaflet import Map, GeoJSON, basemaps

DATA_DIR = Path("data")

with open(DATA_DIR / "regions.geojson") as f:
    regions_fc = json.load(f)

print(f"Loaded {len(regions_fc['features'])} regions.")

## Why Odd/Even Works

Think about it physically. Draw any closed loop on a piece of paper. Pick a point outside the loop and draw a ray in any direction.

- If you're outside and your ray exits the loop: 1 crossing (odd) → you're still outside... wait, that's wrong.

Let's think again. Start at a point far away — clearly outside. Every time the ray crosses the boundary, you toggle between outside and inside:

```text
far away (outside)  →  cross #1  →  inside  →  cross #2  →  outside  →  cross #3  →  inside ...
```

Starting from outside, odd crossings = ended up inside. Even crossings = ended up outside. The ray starts outside and each edge crossing flips the state. This is a consequence of the **Jordan curve theorem** — a closed curve divides the plane into exactly two regions.

We shoot the ray horizontally to the right (positive x direction). This simplifies the edge intersection test.

## The Algorithm Step by Step

```text
input:  point (px, py),  polygon ring [(x0,y0), (x1,y1), ..., (xn,yn), (x0,y0)]
output: True if inside, False if outside

crossings = 0

for each edge (xi,yi) → (xj,yj) in the ring:
    if the edge straddles the horizontal ray from (px, py) going right:
        compute the x coordinate where the edge crosses the ray's y level
        if that x coordinate is to the right of px:
            crossings += 1

return crossings % 2 == 1
```

"Straddles" means one endpoint is above the point's y level and the other is below (or exactly at the level with care for edge cases).

In [ ]:
def point_in_ring(lon, lat, ring):
    """
    Ray casting test: is (lon, lat) inside the closed polygon ring?
    ring is a list of [lon, lat] pairs where ring[0] == ring[-1].
    Returns True if inside, False if outside.
    Boundary behavior is intentionally undefined (may return True or False).
    """
    n = len(ring) - 1   # number of edges (ring[0]==ring[-1], so n pairs)
    inside = False

    for i in range(n):
        xi, yi = ring[i][0],   ring[i][1]
        xj, yj = ring[i+1][0], ring[i+1][1]

        # Does this edge straddle the horizontal ray at y=lat?
        # Exactly one endpoint above, one at or below (< vs >=)
        # This specific comparison handles the vertex-hit edge case
        straddles = (yi > lat) != (yj > lat)

        if straddles:
            # x-coordinate where this edge crosses y=lat
            x_intersect = xi + (lat - yi) * (xj - xi) / (yj - yi)
            # Does the intersection land to the right of our point?
            if lon < x_intersect:
                inside = not inside   # toggle

    return inside


# Quick sanity check on Sector Alpha: [42,34]→[58,34]→[58,42]→[42,42]→[42,34]
alpha_ring = regions_fc["features"][0]["geometry"]["coordinates"][0]

test_cases = [
    (51.388, 35.695, True,  "Tehran — should be inside"),
    (50.0,   38.0,  True,  "Center — should be inside"),
    (30.0,   38.0,  False, "West — should be outside"),
    (70.0,   38.0,  False, "East — should be outside"),
    (50.0,   20.0,  False, "South — should be outside"),
]

print(f"{'Point':<25}  {'Expected':>8}  {'Got':>8}  {'Pass?':>6}")
print("─" * 55)
for lon, lat, expected, label in test_cases:
    result = point_in_ring(lon, lat, alpha_ring)
    ok = "✓" if result == expected else "✗ FAIL"
    print(f"  {label:<23}  {str(expected):>8}  {str(result):>8}  {ok:>6}")

## Handling the Vertex Hit Edge Case

The comparison `(yi > lat) != (yj > lat)` is carefully asymmetric — it uses strict `>` on one side and `>=` implicitly on the other (because Python `!=` with `>` counts one endpoint at exactly `lat` as "below"). This ensures that a ray passing exactly through a vertex is counted only once, not twice (once for each edge that meets at that vertex).

If you use `>=` on both sides, a vertex on the ray gets counted twice — canceling out — and gives the wrong answer for points near a vertex.

In [ ]:
# Demonstrate vertex handling
# A point with the same lat as a vertex of Sector Alpha (lat=34, the bottom edge)
# The ray at lat=34 passes through vertices at (42,34) and (58,34)

vertex_lat = 34.0
test_lons = [40.0, 50.0, 60.0]   # west, inside, east

print("Points with lat = 34.0 (same as bottom edge of Sector Alpha):")
print(f"{'lon':>8}  {'Result':>10}")
for lon in test_lons:
    result = point_in_ring(lon, vertex_lat, alpha_ring)
    print(f"  {lon:>6}     {str(result):>9}")

print()
print("Points on the bottom edge (y=34) may return True or False — boundary behavior.")
print("In practice, real coordinates almost never hit exact polygon vertices.")

## Testing Against the Concave Polygon

The same `point_in_ring` function handles concave shapes correctly — no changes needed. The bounding box check would fail for the notch point; ray casting handles it.

In [ ]:
# Concave polygon from notebook 02
concave_ring = [
    [44, 34], [60, 34], [60, 44],
    [55, 44], [55, 38], [49, 38],
    [49, 44], [44, 44], [44, 34]
]

concave_tests = [
    (52.0, 41.0, "In notch (bbox pass, polygon fail)"),
    (47.0, 37.0, "Clearly inside"),
    (57.0, 41.0, "Inside right arm"),
    (52.0, 35.0, "Inside bottom"),
    (65.0, 40.0, "Outside east"),
]

# Bounding box check (naive approach)
def bbox_check(lon, lat, ring):
    lons = [p[0] for p in ring]
    lats = [p[1] for p in ring]
    return min(lons) <= lon <= max(lons) and min(lats) <= lat <= max(lats)

print(f"{'Point':<35}  {'BBox':>6}  {'RayCast':>8}")
print("─" * 55)
for lon, lat, label in concave_tests:
    bbox = bbox_check(lon, lat, concave_ring)
    raycast = point_in_ring(lon, lat, concave_ring)
    print(f"  {label:<33}  {str(bbox):>6}  {str(raycast):>8}")

print()
print("The notch point: BBox=True (wrong), RayCast=False (correct)")

## Visualizing the Ray

To make the algorithm tangible, we can draw the ray itself as a GeoJSON LineString — from the test point going east until it exits the polygon bounding box.

In [ ]:
def make_ray_feature(lon, lat, length=30):
    """Visualize the horizontal ray from (lon, lat) going east."""
    return {
        "type": "Feature",
        "properties": {"label": f"ray from ({lon},{lat})"},
        "geometry": {
            "type": "LineString",
            "coordinates": [[lon, lat], [lon + length, lat]]
        }
    }

# Show rays from two test points against Sector Alpha
inside_pt  = [51.388, 35.695]   # Tehran — inside
outside_pt = [30.0,   35.695]   # west of sector — outside (same lat)

m = Map(center=(36, 50), zoom=4, basemap=basemaps.CartoDB.Positron)
m.add(GeoJSON(
    data={"type": "FeatureCollection", "features": [regions_fc["features"][0]]},
    style={"color": "#2980b9", "fillColor": "#2980b9", "fillOpacity": 0.15, "weight": 2}
))
m.add(GeoJSON(
    data={"type": "FeatureCollection", "features": [
        make_ray_feature(*inside_pt),
        make_ray_feature(*outside_pt)
    ]},
    style={"color": "#e74c3c", "weight": 2, "opacity": 0.9}
))
m.add(GeoJSON(
    data={"type": "FeatureCollection", "features": [
        {"type": "Feature", "properties": {"n": "inside"},
         "geometry": {"type": "Point", "coordinates": inside_pt}},
        {"type": "Feature", "properties": {"n": "outside"},
         "geometry": {"type": "Point", "coordinates": outside_pt}},
    ]},
    style={"color": "#e74c3c", "fillColor": "#e74c3c", "fillOpacity": 1.0, "weight": 2}
))

in_result  = point_in_ring(*inside_pt, alpha_ring)
out_result = point_in_ring(*outside_pt, alpha_ring)

print(f"Tehran (inside)  → ray crossings: 1 → inside={in_result}")
print(f"West pt (outside) → ray crossings: 0 → inside={out_result}")
m

## Exercise A

Implement `point_in_ring` yourself from scratch — without looking at the teaching cell above.

1. Start from the pseudocode: loop over edges, check if the ray at `lat` is straddled, compute the x-intersection, toggle `inside`.
2. Verify your implementation passes all five assertions below.

In [ ]:
def my_point_in_ring(lon, lat, ring):
    # Your implementation here
    pass

alpha_ring = regions_fc["features"][0]["geometry"]["coordinates"][0]

assert my_point_in_ring(51.388, 35.695, alpha_ring) == True,  "Tehran should be inside Sector Alpha"
assert my_point_in_ring(50.0,   38.0,   alpha_ring) == True,  "Center should be inside"
assert my_point_in_ring(30.0,   38.0,   alpha_ring) == False, "West should be outside"
assert my_point_in_ring(70.0,   38.0,   alpha_ring) == False, "East should be outside"
assert my_point_in_ring(50.0,   20.0,   alpha_ring) == False, "South should be outside"

print("All assertions passed!")

## Exercise B

Use `point_in_ring` to verify your predictions from Exercise A of notebook 02.

For each test city, run the algorithm against the correct sector's ring and print whether it's inside or outside. Were your visual predictions accurate?

In [ ]:
import json
from pathlib import Path

DATA_DIR = Path("data")
with open(DATA_DIR / "regions.geojson") as f:
    regions_fc = json.load(f)

cities = [
    {"name": "Tehran",  "lon": 51.388, "lat": 35.695},
    {"name": "Riyadh",  "lon": 46.675, "lat": 24.688},
    {"name": "Cairo",   "lon": 31.235, "lat": 30.044},
    {"name": "Madrid",  "lon": -3.703, "lat": 40.417},
    {"name": "Muscat",  "lon": 58.593, "lat": 23.614},
]

# For each city, test against every sector
# Print: city name, sector name, inside=True/False
# Report which sector each city falls in (or "none")
# Your code here

---

## Check Your Understanding

**1.** Why does the odd/even crossing count work for determining inside vs outside?

**2.** What is the significance of using `(yi > lat) != (yj > lat)` rather than `(yi >= lat) != (yj >= lat)` in the straddle check?

```python
# No code needed — answer in your own words
```

## Next

In [04 — Testing Against Multiple Features](./04_Testing_Against_Multiple_Features.ipynb), we scale the algorithm up: instead of testing one point against one polygon, we test one point against an entire FeatureCollection and find which feature it belongs to.